# UN Sentiment & Tone Analysis

This analysis examines sentiment and tone in UN Noon Briefings by classifying briefing sections as Positive, Neutral, or Negative and identifying their communicative tone, such as Diplomatic, Concerned, Critical, Urgent, Optimistic, Neutral/Informational, or Constructive. 

# 1. Set up and load data
This notebook uses the pre-uploaded database specified by `BRIEFINGS_DB_PATH`.

Alternatively, the database can be created from scratch for a desired date range:

```bash
briefings update 2026-09-01 2026-09-10

briefings init

briefings load 2026-09-01 2026-09-10
```

In [29]:
import pandas as pd
import ollama
from tqdm import tqdm
import json

from un_llm_analysis.config import BRIEFINGS_DB_PATH
from un_llm_analysis.database import (
    get_db_connection,
    get_briefings_between_dates,
    insert_into_analysis_run,
    insert_sentiment_analysis,
    get_analysis_run,
    get_sentiment_results_between_dates
    
)
from un_llm_analysis.analysis.sentiment import (
    get_existing_sentiment_analysis,
)


In [31]:
import importlib
import un_llm_analysis.analysis.sentiment as sentiment_module

importlib.reload(sentiment_module)

<module 'un_llm_analysis.analysis.sentiment' from 'E:\\correlaid-mentoring-project\\un_llm_analysis\\analysis\\sentiment.py'>

In [40]:
# Get the data in the database with its maximum date range
db_start_date = "2026-09-01"
db_end_date = "2026-09-10"

with get_db_connection(BRIEFINGS_DB_PATH) as conn:
    briefings = get_briefings_between_dates(conn, db_start_date, db_end_date)

2026-10-26 23:44:41.535 | INFO     | un_llm_analysis.database:get_briefings_between_dates:317 - 10 briefings obtained between '2026-09-01' and '2026-09-10'


In [41]:
# Checking the content in the database
conn = get_db_connection()

try:
    cursor = conn.cursor()

    tables = [
        "briefing",
        "section",
        "analysis_run",
        "sentiment_analysis",
    ]

    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        count = cursor.fetchone()[0]
        print(f"{table}: {count} records")

finally:
    conn.close()

briefing: 17 records
section: 136 records
analysis_run: 1 records
sentiment_analysis: 48 records


In [ ]:
# Get the section data from the briefings and store it in a DataFrame

rows = []

for briefing in tqdm(briefings):
    for i, section in enumerate(briefing.sections, start=1):
        rows.append(
            {
                "section_id": section.section_id,
                "briefing_id": section.briefing_id,
                "briefing_date": briefing.briefing_date,
                "section_title": section.title,
                "section_text": section.text,
            }
        )

section_data = pd.DataFrame(rows)

section_data.info()
section_data.tail(5)

In [43]:
# Prompt v3: complete labels and explicit classification rules
prompt_version = "v4"

VALID_SENTIMENTS = {"Positive", "Neutral", "Negative"}
VALID_TONES = {
    "Diplomatic",
    "Concerned",
    "Critical",
    "Urgent",
    "Optimistic",
    "Neutral/Informational",
    "Constructive",
}

ANALYSIS_PROMPT = """
You are classifying a section from a United Nations Noon Briefing.

Classify SENTIMENT using exactly one label:
- Positive: favorable developments, progress, success, support, or hope.
- Neutral: factual reporting without clearly positive or negative evaluation.
- Negative: adverse developments, harm, criticism, failure, or deterioration.

Classify the PRIMARY TONE using exactly one label:
- Diplomatic
- Concerned
- Critical
- Urgent
- Optimistic
- Neutral/Informational
- Constructive

Rules:
- Sentiment and tone are different. A negative situation can be reported neutrally.
- Do not label text Negative only because it mentions conflict or a crisis.
- Base both labels on the actual wording and overall message.
- Choose the single most prominent tone.
- Do not infer emotions that are not supported by the text.

Return only a valid JSON object with exactly these keys:
{"sentiment": "Neutral", "tone": "Neutral/Informational"}

SECTION TITLE:
__SECTION_TITLE__

SECTION TEXT:
__SECTION_TEXT__
""".strip()

print(f"Prompt version: {prompt_version}")
print(ANALYSIS_PROMPT)

Prompt version: v4
You are classifying a section from a United Nations Noon Briefing.

Classify SENTIMENT using exactly one label:
- Positive: favorable developments, progress, success, support, or hope.
- Neutral: factual reporting without clearly positive or negative evaluation.
- Negative: adverse developments, harm, criticism, failure, or deterioration.

Classify the PRIMARY TONE using exactly one label:
- Diplomatic
- Concerned
- Critical
- Urgent
- Optimistic
- Neutral/Informational
- Constructive

Rules:
- Sentiment and tone are different. A negative situation can be reported neutrally.
- Do not label text Negative only because it mentions conflict or a crisis.
- Base both labels on the actual wording and overall message.
- Choose the single most prominent tone.
- Do not infer emotions that are not supported by the text.

Return only a valid JSON object with exactly these keys:
{"sentiment": "Neutral", "tone": "Neutral/Informational"}

SECTION TITLE:
__SECTION_TITLE__

SECTION T

# Install the required Python packages

`pipenv install mlflow openai`

# Open a second terminal in your project folder and run MLflow server

`pipenv run mlflow server --host 127.0.0.1 --port 5000`

In [44]:
import mlflow
from openai import OpenAI

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("sentiment and tone analysis")

# Ollama's OpenAI-compatible API. The API key is required by the SDK,
# but Ollama does not validate this placeholder key.
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

In [46]:

try:
    # Define the model and date range.
    model_name = "llama3.2"
    start_date = "2026-09-01"
    end_date = "2026-09-10"
    conn = get_db_connection()
    models = ["llama3.2"]
    analysis_name = "UN briefing sentiment and tone analysis"
    temperature = 0
    seed = 42
    max_tokens = 256
    # Retrieve existing results for the requested dates and model.
    existing_results = get_existing_sentiment_analysis(
        conn=conn,
        start_date=start_date,
        end_date=end_date,
        model_name=model_name,
    )

    # Skip the analysis if matching results already exist.
    if not existing_results.empty:
        print(
            f"Skipping analysis: {model_name} already has results "
            f"for {start_date} to {end_date}."
        )

    else:
        print(
            f"Starting a new analysis for {model_name}, "
            f"{start_date} to {end_date}."
        )

        # Create a new analysis run.
        analysis_id = insert_into_analysis_run(
            conn=conn,
            analysis_name=analysis_name,
            model_name=model_name,
            model_version=model_name,
            ollama_version=None,
            prompt_version=prompt_version,
            prompt=ANALYSIS_PROMPT,
            temperature=temperature,
        )

        # Initialize counters.
        success_count = 0
        error_count = 0

        # Start MLflow tracking.
        with mlflow.start_run(
            run_name=f"{model_name}-{prompt_version}"
        ):
            mlflow.log_params(
                {
                    "model": model_name,
                    "prompt_version": prompt_version,
                    "temperature": temperature,
                    "seed": seed,
                    "max_tokens": max_tokens,
                    "n_sections": len(section_data),
                    "start_date": start_date,
                    "end_date": end_date,
                }
            )

            # Process each briefing section.
            for _, section in tqdm(
                section_data.iterrows(),
                total=len(section_data),
                desc=f"Classifying with {model_name}",
            ):
                sentiment = None
                tone = None
                status = "error"
                error_message = None

                # Handle missing titles and section text.
                title_value = section.get("section_title")
                text_value = section.get("section_text")

                section_title = (
                    ""
                    if pd.isna(title_value)
                    else str(title_value).strip()
                )

                section_text = (
                    ""
                    if pd.isna(text_value)
                    else str(text_value).strip()
                )

                try:
                    # Check whether section text is available.
                    if not section_text:
                        raise ValueError("Section text is empty.")

                    # Construct the prompt.
                    prompt = ANALYSIS_PROMPT.replace(
                        "__SECTION_TITLE__",
                        section_title or "(no title)",
                    ).replace(
                        "__SECTION_TEXT__",
                        section_text,
                    )

                    # Call the language model.
                    response = client.chat.completions.create(
                        model=model_name,
                        messages=[
                            {
                                "role": "user",
                                "content": prompt,
                            }
                        ],
                        temperature=temperature,
                        seed=seed,
                        max_tokens=max_tokens,
                        response_format={"type": "json_object"},
                    )

                    # Extract the model response.
                    result_text = response.choices[0].message.content

                    if not result_text:
                        raise ValueError(
                            "The model returned an empty response."
                        )

                    # Parse the JSON response.
                    model_result = json.loads(result_text)

                    if not isinstance(model_result, dict):
                        raise ValueError(
                            "Model response must be a JSON object."
                        )

                    # Extract sentiment and tone.
                    sentiment = model_result.get("sentiment")
                    tone = model_result.get("tone")

                    # Validate the sentiment label.
                    if sentiment not in VALID_SENTIMENTS:
                        raise ValueError(
                            f"Invalid sentiment label: {sentiment!r}. "
                            f"Expected one of "
                            f"{sorted(VALID_SENTIMENTS)}"
                        )

                    # Validate the tone label.
                    if tone not in VALID_TONES:
                        raise ValueError(
                            f"Invalid tone label: {tone!r}. "
                            f"Expected one of {sorted(VALID_TONES)}"
                        )

                    # Mark the result as successful.
                    status = "success"
                    success_count += 1

                except Exception as exc:
                    # Record errors without stopping the entire loop.
                    error_message = (
                        f"{type(exc).__name__}: {exc}"
                    )
                    error_count += 1

                    print(
                        f"[ERROR] analysis_id={analysis_id}, "
                        f"section_id={section.get('section_id')}: "
                        f"{error_message}"
                    )

                # Save each result, including failures.
                insert_sentiment_analysis(
                    conn=conn,
                    analysis_id=analysis_id,
                    briefing_id=int(section.briefing_id),
                    section_id=int(section.section_id),
                    briefing_date=section.get("briefing_date"),
                    sentiment=sentiment,
                    tone=tone,
                    status=status,
                    error_message=error_message,
                )

            # Log final metrics after processing all sections.
            mlflow.log_metrics(
                {
                    "successful_sections": success_count,
                    "failed_sections": error_count,
                }
            )

        # Print the final summary.
        print(
            f"Analysis run {analysis_id} finished: "
            f"{success_count} successful, "
            f"{error_count} failed."
        )

finally:
    # Always close the database connection.
    conn.close()
 


Starting a new analysis for llama3.2, 2026-09-01 to 2026-09-10.


Classifying with llama3.2: 100%|██████████| 88/88 [22:29<00:00, 15.34s/it]


🏃 View run llama3.2-v4 at: http://127.0.0.1:5000/#/experiments/1/runs/d5993ad1a47a48c9a254e964771247a3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Analysis run 2 finished: 88 successful, 0 failed.


# Reference



1. This notebook is based on the following notebook.
[Sentiment Analysis Issue Mapping  Notebook](notebooks/franz/02_analyze_issue_mapping.ipynb).
